# eval-shard — chấm 1 checkpoint trên dev175_v11 (Kaggle T4)

Thay `eval_moc.py` (vòng lặp chờ máy train) bằng chấm MỘT checkpoint có sẵn, một
lần, để chạy song song trên nhiều acc Kaggle thay vì tuần tự trên instance thuê.
`params.checkpoint` là tên thư mục con trong dataset adapter (vd `b40`).

In [ ]:
# ORCHESTRA-PARAMS — kaggle_orchestra.py thay cell này lúc đẩy. Giá trị dưới chỉ để chạy tay.
# Chú ý: `tu`/`den` ở đây là CHỈ SỐ CHECKPOINT trong params.checkpoints (mỗi shard = 1
# checkpoint, chấm TRỌN dev175 — không phải khoảng câu như shard_prompt/shard_infer).
SHARD = {"job": "tay", "chi_so": 0, "so_shard": 1, "tu": 0, "den": 1, "commit": "", "noi_tiep": None,
         "output": "diem.jsonl", "checkpoints": ["b40"]}

In [ ]:
import os, sys, shutil, subprocess, json
from pathlib import Path

SRC = next((c.parent.parent.parent for c in Path("/kaggle/input").glob("**/legalqa/bm25.py")
            if "legalqa-v10-src" in str(c)), None)
assert SRC, "Không thấy codebase — gắn dataset legalqa-v10-src"
REPO = Path("/tmp/repo")
shutil.rmtree(REPO, ignore_errors=True)
shutil.copytree(SRC, REPO)
os.chdir(REPO)
sys.path.insert(0, "src"); sys.path.insert(0, "scripts")
commit = (REPO / "COMMIT.txt").read_text().strip()
print("codebase:", commit, "| shard", SHARD)
assert not SHARD["commit"] or commit == SHARD["commit"], f"src {commit} != {SHARD['commit']}"

OUT = Path("/kaggle/working/out"); OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
!pip -q install transformers==5.14.1 peft nltk
!pip -q uninstall -y torchao

In [ ]:
# dev175_v11.jsonl đi CHUNG dataset legalqa-clean (đã có prompt build sẵn bằng
# rerank_v11 + AITeamVN, cùng công thức dev100_v11 — không build lại trên Kaggle).
DEV = next(Path("/kaggle/input").glob("**/dev175_v11.jsonl"), None)
assert DEV, "Không thấy dev175_v11.jsonl — gắn dataset legalqa-clean bản có dev175"

CHECKPOINT = SHARD["checkpoints"][SHARD["tu"]]

BASE = next((p.parent for p in Path("/kaggle/input").glob("**/tokenizer.json")
             if "qwen" in str(p).lower() and (p.parent / "config.json").exists()), None)
assert BASE, "Không thấy model nền Qwen — gắn model qwen3-5-2b-base"
idx = BASE / "model.safetensors.index.json"
can = sorted(set(json.loads(idx.read_text())["weight_map"].values())) if idx.exists() else []
if [n for n in can if not (BASE / n).exists()]:
    FIX = Path("/tmp/base"); FIX.mkdir(exist_ok=True)
    for f in BASE.iterdir():
        if not (FIX / f.name).exists():
            (FIX / f.name).symlink_to(f)
    thuc = sorted(BASE.glob("*.safetensors"))
    assert len(thuc) == len(can), f"{len(thuc)} shard thật vs {len(can)} index cần"
    for ten, that in zip(can, thuc):
        if not (FIX / ten).exists():
            (FIX / ten).symlink_to(that)
    BASE = FIX

ADAPTER = next((q.parent for q in Path("/kaggle/input").glob("**/adapter_config.json")
                if q.parent.name == CHECKPOINT), None)
assert ADAPTER, f"không thấy adapter '{CHECKPOINT}' — gắn dataset legalqa-ckpt-v11"
print("checkpoint", CHECKPOINT, "| base", BASE, "| adapter", ADAPTER, "| dev", DEV)

In [ ]:
# eval_shard.py chấm TRỌN dev175 (--tu 0 --den 0), không cắt theo SHARD["tu"] — chỉ số đó
# đã dùng để chọn CHECKPOINT ở cell trên, không phải khoảng câu.
subprocess.run([sys.executable, "scripts/eval_shard.py",
    "--model", str(BASE), "--adapter", str(ADAPTER), "--dev", str(DEV),
    "--ra", str(OUT), "--tu", "0", "--den", "0",
    "--max-new", str(SHARD.get("max_new", 1024)), "--gen-bs", str(SHARD.get("gen_bs", 2)),
], check=True)

diem_f = OUT / "diem_0_0.json"
diem = json.loads(diem_f.read_text())
ra = OUT / SHARD["output"]
with ra.open("w", encoding="utf-8") as f:
    f.write(json.dumps({"id": CHECKPOINT, **diem}, ensure_ascii=False) + "\n")
print("== GHI", ra, diem)
from kaggle_orchestra import ghi_moc
print("mốc:", ghi_moc(ra, SHARD["tu"], SHARD["den"]))